# RideOps Support - PLAN Section 3 reproducibilityAll numbers below are recomputed from the CSV. September close-time metrics are right-censored.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import StratifiedKFold, cross_val_scorefrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import PipelineROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(ROOT))from src.load import load_ticketsdf = load_tickets(ROOT / 'data' / 'rideops_support_tickets.csv')CHARTS = ROOT / 'charts'CHARTS.mkdir(exist_ok=True)print('Tickets:', len(df), 'Open at 30 Sep:', df.open.sum())

In [ ]:
def snapshot(day):    day = pd.Timestamp(day)    return int(((df.created_at <= day) & (df.final_closed_at.isna() | (df.final_closed_at > day))).sum())dates = pd.to_datetime(['2026-07-09','2026-07-16','2026-07-23','2026-07-30','2026-09-03','2026-09-10','2026-09-17','2026-09-24'])snapshots = pd.Series([snapshot(d) for d in dates], index=dates)print('Weekly open snapshots:', snapshots.to_dict())print('July mean:', round(snapshots.iloc[:4].mean(), 1), 'PLAN 27.5')print('September mean:', round(snapshots.iloc[4:].mean(), 1), 'PLAN 43.3')print('Increase:', round((snapshots.iloc[4:].mean()/snapshots.iloc[:4].mean()-1)*100), '%', 'PLAN +57%')monthly = df.groupby('month').agg(tickets=('ticket_id','size'), escalation_rate=('esc','mean'), mean_close=('res_h','mean'))print('Monthly volume/escalation:', monthly.to_string())print('Escalation rates (%):', (monthly.escalation_rate*100).round(1).to_dict())

In [ ]:
repeat_month = df.groupby('month').repeat14.mean()print('Repeat contact:', (repeat_month*100).round(1).to_dict())print('September repeat mismatch:', round(repeat_month['2026-09']*100, 1), 'vs PLAN 24.8')print('Fastest growing categories, Jul to Sep:')print(pd.crosstab(df.category, df.month)[['2026-07','2026-09']].assign(growth=lambda x: x['2026-09']-x['2026-07']).sort_values('growth', ascending=False).to_string())print('Clients:', df.client_id.nunique())print('Top-10 client share:', round(df.groupby('client_id').size().nlargest(10).sum()/len(df)*100, 1), '%')outcomes = df.loc[df.esc, 'eng_outcome'].value_counts()print('Engineering outcomes:', outcomes.to_dict())print('Avoidable without Fixed:', int((~df.loc[df.esc,'eng_outcome'].eq('Fixed')).sum()), 'bounce:', int(df.bounce.sum()))

In [ ]:
print('Mean close by month:', df.groupby('month').res_h.mean().round(1).to_dict())print('Median close by escalation:', df.groupby('esc').res_h.median().round(1).to_dict())print('Escalated median by month:', df[df.esc].groupby('month').res_h.median().round(1).to_dict())print('Escalated volume share:', round(df.esc.mean()*100,1), '%; hours share:', round(df.loc[df.esc,'res_h'].fillna(0).sum()/df.res_h.fillna(0).sum()*100,1), '%')esc = df[df.esc].copy()handoff = esc.groupby('eng_handoffs').agg(tickets=('ticket_id','size'), median_hours=('res_h','median'), csat=('csat_score','mean'))print('Handoff table:', handoff.round({'median_hours':1,'csat':2}).to_dict('index'))print('Handoffs per escalated:', round(esc.eng_handoffs.mean(),2), 'PLAN 1.6')print('Extra handoffs:', int(esc.eng_handoffs.sum()-len(esc)), 'bounce extra:', int(esc.loc[esc.bounce,'eng_handoffs'].sum()-esc.loc[esc.bounce].shape[0]))print('CSAT escalated/non:', round(df[df.esc].csat_score.mean(),2), round(df[~df.esc].csat_score.mean(),2))print('Reopen CSAT:', round(df[df.reopened].csat_score.mean(),2), 'not reopened:', round(df[~df.reopened].csat_score.mean(),2))

In [ ]:
print('Repeat by category:', (df.groupby('category').repeat14.mean()*100).sort_values(ascending=False).round(1).to_dict())print('GPS and Roster tickets:', int(df.category.isin(['Live tracking / GPS incorrect','Roster / booking not syncing']).sum()))print('GPS and Roster escalations:', int(df[df.category.isin(['Live tracking / GPS incorrect','Roster / booking not syncing'])].esc.sum()))duplicate_month = df[df.esc].groupby('month').eng_outcome.apply(lambda s: s.eq('Duplicate of known issue').mean())print('Duplicate share by escalation month:', (duplicate_month*100).round(1).to_dict())report = df[(df.category=='Report export failing') & df.esc]print('Report export duplicate:', int(report.eng_outcome.eq('Duplicate of known issue').sum()), 'of', len(report))agent = df.groupby('assigned_agent').agg(tickets=('ticket_id','size'), escalation_rate=('esc','mean'), reopen_rate=('reopened','mean'), csat=('csat_score','mean'))print('Agent comparison:', agent.round({'escalation_rate':3,'reopen_rate':3,'csat':2}).to_dict('index'))karan = df[df.assigned_agent=='Karan B']print('Karan reopen:', round(karan.reopened.mean()*100,1), '%; reopened ownership:', int(karan.reopened.sum()), '/', int(df.reopened.sum()))print('Karan first-close median:', round((karan.first_closed_at-karan.created_at).dt.total_seconds().div(3600).median(),1))print('Rahul escalation precision:', round(df[(df.assigned_agent=='Rahul T') & df.esc].eng_outcome.eq('Fixed').mean()*100,1), '%')

In [ ]:
print('Priority close median:', df.groupby('priority').res_h.median().round(1).to_dict())print('Priority escalation:', (df.groupby('priority').esc.mean()*100).round(1).to_dict())print('Priority first-response median:', df.groupby('priority').first_res_h.median().round(2).to_dict())print('Overall first-response median:', round(df.first_res_h.median()*60), 'minutes')print('First response within 4h:', round(df.first_res_h.le(4).mean()*100,1), '%')print('P1 first-response median:', round(df[df.priority=='P1'].first_res_h.median()*60), 'minutes')print('Channel escalation/CSAT:', df.groupby('channel').agg(escalation=('esc','mean'), csat=('csat_score','mean')).round(3).to_dict('index'))print('Tier escalation/CSAT:', df.groupby('client_tier').agg(escalation=('esc','mean'), csat=('csat_score','mean')).round(3).to_dict('index'))

In [ ]:
open_df = df[df.open]print('Open 30 Sep:', len(open_df), open_df.status.value_counts().to_dict())print('Open GPS/Roster:', open_df.category.value_counts().to_dict())print('Open P1 with Engineering:', int(open_df[(open_df.priority=='P1') & open_df.status.eq('Open - with Engineering')].shape[0]))print('Open tickets created:', open_df.created_at.min(), 'to', open_df.created_at.max())print('Open handoffs:', open_df.eng_handoffs.value_counts().sort_index().to_dict())

In [ ]:
esc = df[df.esc].copy()X = (esc.category.fillna('') + ' ' + esc.description.fillna('')).to_numpy()cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)auc = {}for name, y in {'bounce': esc.bounce.astype(int).to_numpy(), 'fixed': esc.eng_outcome.eq('Fixed').astype(int).to_numpy()}.items():    pipe = Pipeline([('tfidf', TfidfVectorizer()), ('model', LogisticRegression(max_iter=1000, random_state=42))])    scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')    auc[name] = scores.mean()    print(name, 'AUC:', round(scores.mean(),3), 'folds:', scores.round(3).tolist())X_all = (df.category.fillna('') + ' ' + df.description.fillna('')).to_numpy()pipe = Pipeline([('tfidf', TfidfVectorizer()), ('model', LogisticRegression(max_iter=1000, random_state=42))])scores = cross_val_score(pipe, X_all, df.esc.astype(int).to_numpy(), cv=cv, scoring='roc_auc')print('Escalated-at-all AUC:', round(scores.mean(),3), 'folds:', scores.round(3).tolist())print('MISMATCH bounce:', round(auc['bounce'],3), 'vs PLAN 0.48')print('MISMATCH fixed:', round(auc['fixed'],3), 'vs PLAN 0.51')

In [ ]:
# 1. Weekly open snapshotsfig, ax = plt.subplots(figsize=(9,4.5))ax.bar([d.strftime('%d %b') for d in snapshots.index], snapshots.values, color=['#82B0D2']*4+['#C44E52']*4)ax.set(title='Weekly open-ticket snapshots', ylabel='Open tickets'); ax.grid(axis='y', alpha=.2)fig.tight_layout(); fig.savefig(CHARTS/'1_weekly_open_snapshots.png', dpi=150); plt.close(fig)# 2. Inflow and median close timemonthly_all = df.groupby('month').agg(inflow=('ticket_id','size'), median_close=('res_h','median'))fig, ax = plt.subplots(figsize=(9,4.5))ax.plot(monthly_all.index, monthly_all.inflow, marker='o', color='#4C72B0', label='Inflow')ax2=ax.twinx(); ax2.plot(monthly_all.index, monthly_all.median_close, marker='o', color='#DD8452', label='Median close')ax.set(title='Inflow and median close time', ylabel='Tickets'); ax2.set_ylabel('Median hours')fig.tight_layout(); fig.savefig(CHARTS/'2_inflow_cycle_time.png', dpi=150); plt.close(fig)# 3. Engineering outcomesorder=['Fixed','Insufficient info - returned to L1','Config issue - returned to L1','Duplicate of known issue','Not a bug - how-to','Not reproducible']values=outcomes.reindex(order).fillna(0)fig, ax=plt.subplots(figsize=(9,5))ax.barh(['Fixed','Insufficient info','Config issue','Known duplicate','How-to','Not reproducible'], values.values, color=['#55A868','#C44E52','#C44E52','#4C72B0','#DD8452','#937860'])ax.invert_yaxis(); ax.set(title='Engineering outcomes', xlabel='Escalated tickets'); ax.grid(axis='x', alpha=.2)fig.tight_layout(); fig.savefig(CHARTS/'3_engineering_outcomes.png', dpi=150); plt.close(fig)# 4. Handoff time and CSATfig, ax=plt.subplots(figsize=(9,4.5))x=[str(v) for v in handoff.index]ax.bar(x, handoff.median_hours, color='#4C72B0')ax2=ax.twinx(); ax2.plot(x, handoff.csat, color='#C44E52', marker='o')ax.set(title='Resolution time and CSAT by handoffs', xlabel='Handoffs', ylabel='Median hours'); ax2.set_ylabel('CSAT')fig.tight_layout(); fig.savefig(CHARTS/'4_handoff_cost_csat.png', dpi=150); plt.close(fig)# 5. Agent reopen ratesagent_sorted=agent.sort_values('reopen_rate', ascending=False)fig, ax=plt.subplots(figsize=(10,5))ax.bar(agent_sorted.index, agent_sorted.reopen_rate*100, color=['#C44E52' if i=='Karan B' else '#4C72B0' for i in agent_sorted.index])ax.set(title='Reopen rate by agent', ylabel='% reopened'); ax.grid(axis='y', alpha=.2)fig.tight_layout(); fig.savefig(CHARTS/'5_agent_reopen_rate.png', dpi=150); plt.close(fig)# 6. Repeat contact by categoryrepeat_category=(df.groupby('category').repeat14.mean()*100).sort_values()fig, ax=plt.subplots(figsize=(9,6))ax.barh(repeat_category.index, repeat_category.values, color='#55A868')ax.set(title='Repeat contact within 14 days', xlabel='% of tickets'); ax.grid(axis='x', alpha=.2)fig.tight_layout(); fig.savefig(CHARTS/'6_repeat_contact.png', dpi=150); plt.close(fig)print('Saved 6 charts to', CHARTS)